In [7]:
import numpy as np
import matplotlib.pyplot as plt

# parameters
INTERVAL_LB = 0
INTERVAL_UB = 1
N = 5

In [20]:
#assuming 1 dimensional, with equally spaced nodes for now
# WIDTH = (INTERVAL_UB - INTERVAL_LB)/N

nodes = np.linspace(INTERVAL_LB,INTERVAL_UB, N+1)
widths = nodes[1:] - nodes[0:-1]


def basis_function(x, lower_node, current_node, upper_node): # lower node and upper node can be left blank in the edge cases 
    # current_node corresponds to the value of j we are checking
    # lower_node is the node before, upper_node is the node after

    # currently generalised to only 1-d cases, idea for multidimensional could be to compare component wise i.e. if x[0] >= current node[0] 
    # and either return standardised euclidean distance or simply only standardised distance from selected compent i.e. (upper_node[0]-x[0])/(upper_node[0]-current_node[0])

    if x >= current_node: 
        assert upper_node != np.nan, "Upper Node has not been given (likely out of bounds error)" # controls the error

        if x <= upper_node: 
            return (upper_node-x)/(upper_node-current_node)
        
    if x <= current_node: 
        assert lower_node != np.nan, "Lower Node has not been given (likely out of bounds error)"

        if x >= lower_node:
            return (x-lower_node)/(current_node-lower_node)

    return 0.0


def phi(x, nodes): # aggregates basis functions over all nodes

    extended_nodes = np.concat([np.array([np.nan]),nodes, np.array([np.nan])])  # pads the sides with nans

    lower_nodes = extended_nodes[:-2] 
    current_nodes = extended_nodes[1:-1] 
    upper_nodes = extended_nodes[2:]
    
    return np.vectorize(basis_function) (x, lower_nodes, current_nodes, upper_nodes)

x = 0.5910573
phi_x = phi(x,nodes)
print(phi_x)


[0.        0.        0.0447135 0.9552865 0.        0.       ]


In [34]:
T = np.zeros([N+1,N+1]) # mass matrix
S = np.zeros_like(T)    # stiffness matrix

# tridiagonal matrices
# from calculations shown in the paper, will need to be analysed before extending model

for i in range(0,N+1):
    padded_widths = np.concat([widths,np.zeros(1)]) # having widths[N] = widths[-1] = 0 simplifies the boundary cases for T[i][i] 

    T[i][i] = (padded_widths[i]+padded_widths[i-1])/3 

    if i != 0:
        S[i][i] += 1/padded_widths[i-1] # left hand component of S[i][i], non-zero for all i != 0

        T[i][i-1] = padded_widths[i-1]/6
        S[i][i-1] = -1/padded_widths[i-1] 

        T[i-1][i] = padded_widths[i-1]/6
        S[i-1][i] = -1/padded_widths[i-1]
    if i != N:
        S[i][i] += 1/padded_widths[i]   # right hand component of S[i][i], non-zero for all i != N


print(T)
print(S)

[[0.06666667 0.03333333 0.         0.         0.         0.        ]
 [0.03333333 0.13333333 0.03333333 0.         0.         0.        ]
 [0.         0.03333333 0.13333333 0.03333333 0.         0.        ]
 [0.         0.         0.03333333 0.13333333 0.03333333 0.        ]
 [0.         0.         0.         0.03333333 0.13333333 0.03333333]
 [0.         0.         0.         0.         0.03333333 0.06666667]]
[[ 5. -5.  0.  0.  0.  0.]
 [-5. 10. -5.  0.  0.  0.]
 [ 0. -5. 10. -5.  0.  0.]
 [ 0.  0. -5. 10. -5.  0.]
 [ 0.  0.  0. -5. 10. -5.]
 [ 0.  0.  0.  0. -5.  5.]]


In [10]:
# Firstly assume no damping, so No first order differential term

